<a href="https://colab.research.google.com/github/danybereche-ctrl/ciencia-de-datos-/blob/main/automatizacionsemana6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install transformers scikit-learn pandas

In [8]:
import pandas as pd
import numpy as np
# Datos ficticios para las noticias
noticias_data = {
    'texto': [
        'El Banco Central anunció nuevas medidas para controlar la inflación.',
        'El equipo local ganó el campeonato con un marcador de 3 a 1.',
        'Un nuevo estudio revela los beneficios de la dieta mediterránea para el corazón.',
        'La bolsa de valores experimentó una caída significativa esta semana.',
        'El delantero estrella se recupera de una lesión y volverá a jugar pronto.',
        'Campañas de vacunación masiva se implementan en varias regiones.',
        'Incremento en el precio del petróleo afecta los mercados globales.',
        'Atleta olímpico rompe récord mundial en la competencia de salto largo.',
        'Consejos para mantener una buena salud mental durante el confinamiento.',
        'El gobierno aprueba un paquete de estímulo económico para pequeñas empresas.'
    ],
    'categoria_real': [
        'economía', 'deportes', 'salud', 'economía', 'deportes', 'salud', 'economía', 'deportes', 'salud', 'economía'
    ]
}
# Crear el DataFrame
df_noticias = pd.DataFrame(noticias_data)
# Guardar el DataFrame en un archivo CSV
df_noticias.to_csv('noticias.csv', index=False)
print('Archivo noticias.csv creado exitosamente.')
df_noticias


Archivo noticias.csv creado exitosamente.


,texto,categoria_real
0,El Banco Central anunció nuevas medidas para c...,economía
1,El equipo local ganó el campeonato con un marc...,deportes
2,Un nuevo estudio revela los beneficios de la d...,salud
3,La bolsa de valores experimentó una caída sign...,economía
4,El delantero estrella se recupera de una lesió...,deportes
5,Campañas de vacunación masiva se implementan e...,salud
6,Incremento en el precio del petróleo afecta lo...,economía
7,Atleta olímpico rompe récord mundial en la com...,deportes
8,Consejos para mantener una buena salud mental ...,salud
9,El gobierno aprueba un paquete de estímulo eco...,economía


In [11]:
df_noticias[['texto', 'categoria_real']]

,texto,categoria_real
0,El Banco Central anunció nuevas medidas para c...,economía
1,El equipo local ganó el campeonato con un marc...,deportes
2,Un nuevo estudio revela los beneficios de la d...,salud
3,La bolsa de valores experimentó una caída sign...,economía
4,El delantero estrella se recupera de una lesió...,deportes
5,Campañas de vacunación masiva se implementan e...,salud
6,Incremento en el precio del petróleo afecta lo...,economía
7,Atleta olímpico rompe récord mundial en la com...,deportes
8,Consejos para mantener una buena salud mental ...,salud
9,El gobierno aprueba un paquete de estímulo eco...,economía


In [12]:
from transformers import pipeline
clasificador = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
print("Modelo cargado correctamente")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Modelo cargado correctamente


In [13]:
etiquetas = [ 'econimía', 'deporte', 'salud']

In [18]:
etiquetas [2]



'salud'

In [19]:
resultado = clasificador(
    "El dólar aumentó durante la semana",
    etiquetas
)
resultado

{'sequence': 'El dólar aumentó durante la semana',
 'labels': ['econimía', 'deporte', 'salud'],
 'scores': [0.6007835268974304, 0.21660485863685608, 0.18261165916919708]}

In [20]:
resultado = clasificador(
    "El banco central informo que el dolar aumento por el crecimiento de la inflación",
    etiquetas
)
resultado

{'sequence': 'El banco central informo que el dolar aumento por el crecimiento de la inflación',
 'labels': ['econimía', 'deporte', 'salud'],
 'scores': [0.8669125437736511, 0.07977136224508286, 0.05331609770655632]}

Interpretación

In [21]:
resultado['labels'][2]

'salud'

In [22]:
def clasificar_texto(texto):
    resultado = clasificador(texto,etiquetas)
    categoria = resultado["labels"][0]
    confianza = round(resultado["scores"][0] * 100,2)
    return categoria, confianza

In [23]:
clasificar_texto('El dolar aumento durante la semana')

('econimía', 56.13)

In [25]:
df_noticias[ ["categoria_predicha","confianza_%"]] = df_noticias["texto"].apply(
    lambda x:
    pd.Series(
        clasificar_texto(x)
    )
)

In [26]:
df_noticias

,texto,categoria_real,categoria_predicha,confianza_%
0,El Banco Central anunció nuevas medidas para c...,economía,econimía,78.71
1,El equipo local ganó el campeonato con un marc...,deportes,deporte,50.73
2,Un nuevo estudio revela los beneficios de la d...,salud,salud,98.71
3,La bolsa de valores experimentó una caída sign...,economía,econimía,86.73
4,El delantero estrella se recupera de una lesió...,deportes,deporte,70.54
5,Campañas de vacunación masiva se implementan e...,salud,salud,46.33
6,Incremento en el precio del petróleo afecta lo...,economía,econimía,81.81
7,Atleta olímpico rompe récord mundial en la com...,deportes,deporte,86.33
8,Consejos para mantener una buena salud mental ...,salud,salud,95.79
9,El gobierno aprueba un paquete de estímulo eco...,economía,econimía,93.50


In [28]:
df_noticias.to_csv("noticias_clasificadas.csv",index=False)

In [29]:
df=pd.read_csv("noticias_clasificadas.csv")
df

,texto,categoria_real,categoria_predicha,confianza_%
0,El Banco Central anunció nuevas medidas para c...,economía,econimía,78.71
1,El equipo local ganó el campeonato con un marc...,deportes,deporte,50.73
2,Un nuevo estudio revela los beneficios de la d...,salud,salud,98.71
3,La bolsa de valores experimentó una caída sign...,economía,econimía,86.73
4,El delantero estrella se recupera de una lesió...,deportes,deporte,70.54
5,Campañas de vacunación masiva se implementan e...,salud,salud,46.33
6,Incremento en el precio del petróleo afecta lo...,economía,econimía,81.81
7,Atleta olímpico rompe récord mundial en la com...,deportes,deporte,86.33
8,Consejos para mantener una buena salud mental ...,salud,salud,95.79
9,El gobierno aprueba un paquete de estímulo eco...,economía,econimía,93.50


In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [32]:
y_real = df_noticias["categoria_real"]
y_predicho = df_noticias["categoria_predicha"]





In [35]:
accuracy = accuracy_score(y_real,
y_predicho
)

In [36]:
precision = precision_score(
    y_real,
    y_predicho,
    average="weighted"
)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [38]:

accuracy = accuracy_score(y_real,
y_predicho
)


precision = precision_score(
    y_real,
    y_predicho,
    average="weighted"
)


recall = recall_score(
    y_real,
    y_predicho,
    average="weighted"
)


f1 = f1_score(
    y_real,
    y_predicho,
    average="weighted"
)

print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall: {recall*100:.2f}%")
print(f"F1 Score: {f1*100:.2f}%")



Accuracy: 30.00%
Precision: 30.00%
Recall: 30.00%
F1 Score: 30.00%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [39]:
print(
    classification_report(
        y_real,
        y_predicho
    )
)

              precision    recall  f1-score   support

     deporte       0.00      0.00      0.00         0
    deportes       0.00      0.00      0.00         3
    econimía       0.00      0.00      0.00         0
    economía       0.00      0.00      0.00         4
       salud       1.00      1.00      1.00         3

    accuracy                           0.30        10
   macro avg       0.20      0.20      0.20        10
weighted avg       0.30      0.30      0.30        10



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_